In [3]:
DEEPSEEK_API_KEY = 'sk-135fe459060d4443ab30b8ae1f68f900'

In [17]:
import json
from openai import OpenAI
from datetime import datetime

client = OpenAI(
    api_key= DEEPSEEK_API_KEY ,
    base_url="https://api.deepseek.com",
)

system_prompt = """
```xml
<instruction>
你是一个求职助手，需要从用户的求职需求中提取出关键信息，并以结构化的 JSON 格式返回结果。请按照以下步骤完成任务：

1. 阅读并理解用户提供的求职需求内容（在 <input> 标签中）。
2. 从中提取出以下关键信息：
    - 意向公司（intended_company):
        -这是用户希望应聘的公司名称，可以有多个，请以数组形式返回
        -如果用户没有明确提到任何公司，请返回空数组 []
    - 意向行业（intended_industry):
        - 这是用户希望应聘的公司行业，可以有多个，请以数组形式返回
        - 如果用户没有明确提到任何意向行业，请返回空数组
    - 意向岗位（intended_position）：
        -这是用户有可能希望应聘的多个职位名称，请以数组形式呈现
        -用户有可能明确提出自己希望应聘的职位，也有可能提及意向的工作内容, 也有可能提及自己所擅长的技能或所学专业。请根据用户的输入，尽量返回更多的相关意向岗位。
    - 求职类型（job_type）：
        -单个求职类型为 “社招|校招|实习” 中的一种，用户可以同时寻求多种类型的职位，所以请返回一个包含一个或多个求职类型的数组
        -请确保返回的数组中内容不重复
        -用户如果明确提出了求职类型需求，就请严格按照用户的需求返回
        -用户如果提及自己有正式工作经历（不包含实习），请返回 "["社招"]"
        -用户如果没有明确提出求职类型需求，但是提到了自己的毕业年份或现在距离毕业有多长时间，就请根据当前时间 {curr_date} 猜测求职类型：
            - 如果当前距离毕业时间还有一年以内或刚过毕业时间一年以内，请返回 "["校招", "实习"]"
            - 如果当前距离毕业时间还有一年以上，请返回 "["实习"]"
            - 如果当前已经比毕业时间晚了一年以上，请返回 "["社招"]"
        -用户如果没有提及任何相关信息，请返回 "["社招"]"
3. 以 JSON 格式输出结果，确保字段名称与上述变量名一致。
4. 输出中**不要包含任何 XML 标签**，只返回结构化的 JSON 内容。

注意：
- 如果某项信息未在输入中明确提及，请将其值设为空（字符串字段）或空数组（数组字段）。
</instruction>
```
"""

user_prompt = "我今年就要毕业了，想找数据科学相关的工作"

messages = [{"role": "system", "content": system_prompt.format(curr_date= datetime.today().strftime('%Y-%m-%d'))},
            {"role": "user", "content": user_prompt}]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    response_format={
        'type': 'json_object'
    }
)

print(json.loads(response.choices[0].message.content))

{'intended_company': [], 'intended_industry': [], 'intended_position': ['数据科学'], 'job_type': ['校招', '实习']}


In [15]:
from datetime import datetime

datetime.today().strftime('%Y-%m-%d')

'2025-07-25'

In [9]:
response.choices[0].message.content

'{\n    "question": "Which is the longest river in the world?",\n    "answer": "The Nile River"\n}'